In [6]:
from frag_gnn.runner import load_config
from frag_gnn.pl_model import FragGNNPL, NeimsPL
from frag_gnn.massformer.pl_model import MassFormerPL
from frag_gnn.iceberg.pl_model import IcebergIntenPL, IcebergGenPL
from frag_gnn.graff.pl_model import GrAFFPL
from frag_gnn.utils.misc_utils import NestedDefaultDict, booltype

import argparse
from pprint import pprint
import pytorch_lightning as pl
from IPython.display import display
import os

In [7]:
os.chdir("/h/adamo/frag-gnn")

In [11]:
model_type_to_model_cls = {
    "neims_long": NeimsPL,
    "massformer_long": MassFormerPL,
    "fraggnn_d3_long": FragGNNPL,
    "fraggnn_d4_long": FragGNNPL,
    "iceberg_gen": IcebergGenPL,
    "iceberg_inten3_100_long": IcebergIntenPL,
    "iceberg_inten3_all_long": IcebergIntenPL,
    "graff3_long": GrAFFPL
}
model_types = list(model_type_to_model_cls.keys())

model_to_config = NestedDefaultDict()

# setup conigs
template_fp = "config/template.yml"
for model_type in model_types:
    model_to_config[model_type] = f"config/nist20v3-1_d4_wr00_inchikey/{model_type}/s0.yml"
pprint(model_to_config)

{'neims_long': 'config/nist20v3-1_d4_wr00_inchikey/neims_long/s0.yml', 'massformer_long': 'config/nist20v3-1_d4_wr00_inchikey/massformer_long/s0.yml', 'fraggnn_d3_long': 'config/nist20v3-1_d4_wr00_inchikey/fraggnn_d3_long/s0.yml', 'fraggnn_d4_long': 'config/nist20v3-1_d4_wr00_inchikey/fraggnn_d4_long/s0.yml', 'iceberg_gen': 'config/nist20v3-1_d4_wr00_inchikey/iceberg_gen/s0.yml', 'iceberg_inten3_100_long': 'config/nist20v3-1_d4_wr00_inchikey/iceberg_inten3_100_long/s0.yml', 'iceberg_inten3_all_long': 'config/nist20v3-1_d4_wr00_inchikey/iceberg_inten3_all_long/s0.yml', 'graff3_long': 'config/nist20v3-1_d4_wr00_inchikey/graff3_long/s0.yml'}


In [28]:
model_to_param_counts = NestedDefaultDict()
model_to_summary = {}

for model_type in model_types:
    print(f">>> Starting model_type {model_type}")
    custom_fp = model_to_config[model_type]
    print(f">>> Loading config {custom_fp}")
    config_d = load_config(template_fp, custom_fp)
    config_d["accelerator"] = "cpu"
    config_d["compile"] = False
    print(f">>> Initializing model")
    model_cls = model_type_to_model_cls[model_type]
    model = model_cls(**config_d)
    if model_type in ["neims_long","massformer_long"]:
        model_summary = pl.utilities.model_summary.ModelSummary(model, max_depth=4)
        model_to_summary[model_type] = model_summary
        model_idx = model_summary.layer_names.index("model")
        output_idxs = [model_summary.layer_names.index(f"model.ffn.{k}") for k in ["forw_out_layer","rev_out_layer","out_gate"]]
        model_to_param_counts[model_type] = model_summary.param_nums[model_idx]
        model_to_param_counts[f"{model_type}_output"] = \
            model_summary.param_nums[model_idx] - sum([model_summary.param_nums[i] for i in output_idxs])
    else:
        model_summary = pl.utilities.model_summary.ModelSummary(model, max_depth=4)
        model_to_summary[model_type] = model_summary
        model_idx = model_summary.layer_names.index("model")
        model_to_param_counts[model_type] = model_summary.param_nums[model_idx]
    del model

if "iceberg_inten3_100_long" in model_to_param_counts:
    assert "iceberg_gen" in model_to_param_counts
    model_to_param_counts["iceberg_inten3_100_long"] += model_to_param_counts.pop("iceberg_gen")

if "neims_long" in model_to_param_counts:
    model_to_param_counts["neims_long_nooutput"] = model_to_param_counts["neims_long"] - model_to_param_counts["neims_long_output"]
    del model_to_param_counts["neims_long_output"]

if "massformer_long" in model_to_param_counts:
    model_to_param_counts["massformer_long_nooutput"] = model_to_param_counts["massformer_long"] - model_to_param_counts["massformer_long_output"]
    del model_to_param_counts["massformer_long_output"]


>>> Starting model_type neims_long
>>> Loading config config/nist20v3-1_d4_wr00_inchikey/neims_long/s0.yml
>>> Initializing model
>>> Starting model_type massformer_long
>>> Loading config config/nist20v3-1_d4_wr00_inchikey/massformer_long/s0.yml
>>> Initializing model
>>> Starting model_type fraggnn_d3_long
>>> Loading config config/nist20v3-1_d4_wr00_inchikey/fraggnn_d3_long/s0.yml
>>> Initializing model
>>> Starting model_type fraggnn_d4_long
>>> Loading config config/nist20v3-1_d4_wr00_inchikey/fraggnn_d4_long/s0.yml
>>> Initializing model
>>> Starting model_type iceberg_gen
>>> Loading config config/nist20v3-1_d4_wr00_inchikey/iceberg_gen/s0.yml
>>> Initializing model
>>> Starting model_type iceberg_inten3_100_long
>>> Loading config config/nist20v3-1_d4_wr00_inchikey/iceberg_inten3_100_long/s0.yml
>>> Initializing model
>>> Starting model_type iceberg_inten3_all_long
>>> Loading config config/nist20v3-1_d4_wr00_inchikey/iceberg_inten3_all_long/s0.yml
>>> Initializing model
>>> St

In [33]:
model_to_summary["massformer_long"]

   | Name                                      | Type                           | Params
----------------------------------------------------------------------------------------------
0  | model                                     | MassFormerModel                | 165 M 
1  | model.gf                                  | GFv2Embedder                   | 48.3 M
2  | model.gf.encoder                          | GraphormerEncoder              | 48.3 M
3  | model.gf.encoder.graph_encoder            | GraphormerGraphEncoder         | 47.1 M
4  | model.gf.encoder.masked_lm_pooler         | Linear                         | 590 K 
5  | model.gf.encoder.lm_head_transform_weight | Linear                         | 590 K 
6  | model.gf.encoder.layer_norm               | LRPLayerNorm                   | 1.5 K 
7  | model.ce_embedder                         | Sequential                     | 1.9 K 
8  | model.ce_embedder.0                       | FourierFeaturizerAbsoluteSines | 1.6 K 
9  | model.ce_e

In [19]:
model_to_param_counts_m = {k: f"{v/1e6:.1f}" for k,v in model_to_param_counts.items()}


pprint(model_to_param_counts_m)

{'fraggnn_d3_long': '1.2',
 'fraggnn_d4_long': '1.6',
 'graff3_long': '57.8',
 'iceberg_inten3_100_long': '17.6',
 'iceberg_inten3_all_long': '13.5',
 'massformer_long': '165.3',
 'massformer_long_nooutput': '116.0',
 'neims_long': '125.4',
 'neims_long_nooutput': '116.4'}
